# Trabajo Práctico Final — Sistema de Detección y Segmentación de Objetos en Tiempo Real
## Notebook 03 · Evaluación del modelo

**Maestría en Inteligencia Artificial — Deep Learning**

---

Este notebook implementa la tercera etapa del pipeline: la **evaluación del modelo ajustado** sobre la partición de prueba, reservada desde el Notebook 01 y no utilizada en ninguna decisión de entrenamiento. Comprende:

1. Definición del protocolo de evaluación y de las métricas estándar (IoU, mAP).
2. Métricas globales y por clase, para detección (cajas) y segmentación (máscaras).
3. Matriz de confusión y curvas precisión–recall.
4. Comparación cuantitativa con el modelo preentrenado de referencia (*baseline*).
5. Análisis cualitativo de predicciones y de casos de error.
6. Medición de la velocidad de inferencia.

> **Nota de ejecución:** ejecutar en el servidor GPU, tras completar los Notebooks 01 y 02.

In [ ]:
import random
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import numpy as np
import pandas as pd
from ultralytics import YOLO

SEED = 42
random.seed(SEED)

CLASSES = ['person', 'cell phone', 'cup', 'bottle', 'laptop', 'keyboard', 'mouse', 'book']

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
DATA_DIR = ROOT / 'data' / 'coco_subset'
DATA80_DIR = ROOT / 'data' / 'coco_subset_80'
RUNS_DIR = ROOT / 'runs'
PESOS = ROOT / 'models' / 'yolov8n_seg_best.pt'

assert PESOS.exists(), 'No se encontraron los pesos: ejecutar primero el Notebook 02.'

# Estilo de los gráficos
C_AZUL, C_AQUA = '#2a78d6', '#1baf7a'
plt.rcParams.update({
    'figure.facecolor': '#fcfcfb', 'axes.facecolor': '#fcfcfb',
    'axes.edgecolor': '#c3c2b7', 'axes.grid': True, 'axes.axisbelow': True,
    'grid.color': '#e1e0d9', 'grid.linewidth': 0.8,
    'axes.titlesize': 12, 'font.family': 'sans-serif',
})

model = YOLO(str(PESOS))
print('Modelo cargado:', PESOS.name)

### 1. Protocolo de evaluación y métricas

**IoU (Intersection over Union).** Mide el solapamiento entre una predicción $P$ y su anotación de referencia $G$:

$$IoU(P, G) = \frac{|P \cap G|}{|P \cup G|}$$

Se aplica tanto a cajas delimitadoras (áreas rectangulares) como a máscaras (conjuntos de píxeles). Una predicción se considera **verdadero positivo** si su IoU con una anotación de la misma clase supera un umbral dado; en caso contrario es un falso positivo.

**Precisión y recall.** Fijado un umbral de IoU y ordenadas las predicciones por confianza, la precisión mide qué fracción de las predicciones es correcta y el recall qué fracción de los objetos reales fue encontrada. La **AP (Average Precision)** de una clase es el área bajo su curva precisión–recall.

**mAP (mean Average Precision).** Promedio de la AP sobre las clases. Se reportan las dos variantes estándar del protocolo COCO:

- **mAP@50** — con umbral de IoU = 0,5 (criterio laxo: premia encontrar y clasificar bien los objetos).
- **mAP@50-95** — promedio sobre umbrales de IoU de 0,5 a 0,95 en pasos de 0,05 (criterio estricto: premia además la precisión de la localización). Es la métrica principal de este trabajo.

Cada métrica se calcula por separado para **cajas** (detección) y para **máscaras** (segmentación), de modo que se evalúan explícitamente ambas tareas requeridas por la consigna.

In [ ]:
metrics = model.val(
    data=str(DATA_DIR / 'dataset.yaml'),
    split='test',            # evaluación sobre la partición de prueba
    project=str(RUNS_DIR),
    name='eval_test',
    exist_ok=True,
    plots=True,
)

In [ ]:
resumen = pd.DataFrame({
    'Cajas (detección)': [metrics.box.mp, metrics.box.mr, metrics.box.map50, metrics.box.map],
    'Máscaras (segmentación)': [metrics.seg.mp, metrics.seg.mr, metrics.seg.map50, metrics.seg.map],
}, index=['Precisión media', 'Recall medio', 'mAP@50', 'mAP@50-95']).round(3)
resumen

### 2. Resultados por clase

El desglose por clase permite identificar fortalezas y debilidades del modelo. Cabe esperar mejor desempeño en clases con más ejemplos de entrenamiento (*person*) y en objetos de contorno regular; los objetos pequeños o frecuentemente ocluidos (*mouse*, *cell phone*) suelen presentar mAP inferior, en particular en la métrica estricta de máscaras.

In [ ]:
nombres = [metrics.names[i] for i in range(len(metrics.names))]
por_clase = pd.DataFrame({
    'mAP@50-95 cajas': metrics.box.maps,
    'mAP@50-95 máscaras': metrics.seg.maps,
}, index=nombres).round(3)

x = np.arange(len(nombres))
w = 0.36
fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(x - w / 2, por_clase['mAP@50-95 cajas'], width=w, label='Cajas (detección)', color=C_AZUL)
ax.bar(x + w / 2, por_clase['mAP@50-95 máscaras'], width=w, label='Máscaras (segmentación)', color=C_AQUA)
ax.set_xticks(x)
ax.set_xticklabels(nombres, rotation=30, ha='right')
ax.set_ylabel('mAP@50-95 (test)')
ax.set_title('Desempeño por clase en la partición de prueba')
ax.grid(axis='x', visible=False)
ax.legend()
plt.tight_layout()
plt.show()

por_clase

### 3. Matriz de confusión y curvas precisión–recall

La rutina de validación genera automáticamente la **matriz de confusión normalizada** (que expone confusiones sistemáticas entre clases y la proporción de objetos no detectados, columna *background*) y las **curvas precisión–recall** por clase, cuya área es la AP reportada.

In [ ]:
figuras = ['confusion_matrix_normalized.png', 'MaskPR_curve.png']
fig, axes = plt.subplots(1, 2, figsize=(17, 7))
for ax, nombre in zip(axes, figuras):
    ruta = RUNS_DIR / 'eval_test' / nombre
    ax.imshow(mpimg.imread(ruta))
    ax.set_axis_off()
    ax.set_title(nombre.replace('.png', ''))
plt.tight_layout()
plt.show()

### 4. Comparación con el modelo de referencia (baseline)

Para cuantificar el aporte del ajuste fino se evalúa el modelo **preentrenado original** (`yolov8n-seg.pt`, 80 clases) sobre la **misma partición de prueba**, utilizando la copia exportada con el vocabulario completo de 80 clases (Notebook 01, sección 7), de modo que los índices de clase de las etiquetas coincidan con los del modelo.

De los resultados del baseline se extraen las AP de las ocho clases de interés y se comparan con las del modelo ajustado.

> **Consideración metodológica:** la comparación es conservadora respecto del baseline — el modelo preentrenado resuelve un problema más difícil (80 clases posibles en lugar de 8), por lo que parte de la mejora observada se explica por la reducción del vocabulario además de por el ajuste de los pesos. Esta salvedad se hace explícita en el reporte técnico.

In [ ]:
baseline = YOLO('yolov8n-seg.pt')
metrics_base = baseline.val(
    data=str(DATA80_DIR / 'dataset.yaml'),
    split='val',             # esta clave contiene la partición de PRUEBA (ver Notebook 01, sección 7)
    project=str(RUNS_DIR),
    name='eval_baseline',
    exist_ok=True,
)

idx_por_nombre = {v: k for k, v in metrics_base.names.items()}
idx = [idx_por_nombre[c] for c in CLASSES]

comparacion = pd.DataFrame({
    'Baseline preentrenado': [float(metrics_base.seg.maps[i]) for i in idx],
    'Fine-tuned (TPF)': [float(por_clase.loc[c, 'mAP@50-95 máscaras']) for c in CLASSES],
}, index=CLASSES).round(3)
comparacion.loc['— promedio —'] = comparacion.mean().round(3)
comparacion

In [ ]:
datos = comparacion.drop(index='— promedio —')
x = np.arange(len(datos.index))
w = 0.36
fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(x - w / 2, datos['Baseline preentrenado'], width=w, label='Baseline preentrenado', color=C_AZUL)
ax.bar(x + w / 2, datos['Fine-tuned (TPF)'], width=w, label='Fine-tuned (TPF)', color=C_AQUA)
ax.set_xticks(x)
ax.set_xticklabels(datos.index, rotation=30, ha='right')
ax.set_ylabel('mAP@50-95 de máscaras (test)')
ax.set_title('Aporte del ajuste fino por clase')
ax.grid(axis='x', visible=False)
ax.legend()
plt.tight_layout()
plt.show()

### 5. Análisis cualitativo

Se contrastan las predicciones de ambos modelos sobre imágenes de la partición de prueba. Además de confirmar visualmente las métricas, este análisis permite observar el efecto del vocabulario reducido: el baseline dispersa confianza entre clases fuera del alcance del sistema, mientras que el modelo ajustado concentra sus predicciones en las ocho clases de interés.

Para el reporte técnico conviene ejecutar esta celda varias veces (cambiando la muestra) y seleccionar tanto **casos de éxito** como **casos de error** (oclusiones, objetos pequeños, instancias superpuestas), que fundamentan la sección de análisis de limitaciones.

In [ ]:
imgs_test = sorted((DATA_DIR / 'images' / 'test').glob('*.jpg'))
muestra = random.sample(imgs_test, 4)
modelos = [(baseline, 'Baseline (80 clases)'), (model, 'Fine-tuned (8 clases)')]

fig, axes = plt.subplots(2, 4, figsize=(18, 8.5))
for j, img_path in enumerate(muestra):
    for i, (m, titulo) in enumerate(modelos):
        r = m.predict(str(img_path), conf=0.4, verbose=False)[0]
        axes[i, j].imshow(cv2.cvtColor(r.plot(), cv2.COLOR_BGR2RGB))
        axes[i, j].set_axis_off()
for i, (_, titulo) in enumerate(modelos):
    axes[i, 0].text(-0.04, 0.5, titulo, transform=axes[i, 0].transAxes,
                    rotation=90, va='center', ha='right', fontsize=12)
fig.suptitle('Predicciones sobre la partición de prueba: baseline vs. modelo ajustado', y=0.99)
plt.tight_layout()
plt.show()

### 6. Velocidad de inferencia

El requisito de **inferencia en tiempo real** exige cuantificar la latencia. La rutina de validación reporta el tiempo medio por imagen desglosado en preprocesamiento, inferencia y posprocesamiento, medido aquí sobre la **GPU del servidor**.

La latencia relevante para la demostración final es, no obstante, la de la **CPU de la computadora personal** donde se ejecuta `src/demo_webcam.py`; esa medición (fotogramas por segundo efectivos, con y sin optimización OpenVINO) se realiza localmente y se incorpora al reporte en la sección del sistema de inferencia.

In [ ]:
velocidad = pd.Series(metrics.speed, name='ms por imagen (GPU)').round(2)
print(velocidad.to_string())
lat_total = sum(metrics.speed.values())
print(f'\nLatencia total: {lat_total:.1f} ms/imagen  →  {1000 / lat_total:.0f} FPS teóricos en GPU')

### 7. Síntesis

*(Completar con los valores obtenidos tras la ejecución.)*

- El modelo ajustado alcanzó un **mAP@50-95 de ___ (cajas) y ___ (máscaras)** sobre la partición de prueba, frente a ___ y ___ del baseline preentrenado en las mismas ocho clases.
- Las clases de mejor desempeño fueron ___; las de peor desempeño, ___, atribuible a ___ (tamaño de los objetos, oclusión, cantidad de ejemplos).
- La matriz de confusión muestra ___ (confusiones sistemáticas relevantes o su ausencia).
- La latencia en GPU fue de ___ ms/imagen; la medición en CPU local se reporta junto con la demostración.

**Cierre del pipeline:** descargar del servidor `models/yolov8n_seg_best.pt`, el directorio `runs/` (curvas y figuras para el reporte) y los notebooks ejecutados. La demostración en tiempo real se ejecuta localmente con `python src/demo_webcam.py`.